<p align="center">
    <span style="font-size:2.5em; font-weight:bold;">
        eFleetPlan - Optimal infrastructure and fleet operation of electric LCV
    </span>
</p>

<p align="center">
    <span style="font-size:1.5em; font-weight:bold;">
        Carolina Gil Ribeiro, Jagruti Thakur
    </span>
</p>

## 0. Importing dependencies

In [ ]:
from pathlib import Path
import sys
import time
from glob import glob

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


def find_project_root(start: Path) -> Path:
    """
    Find project root by looking for common project markers.
    """
    for parent in [start, *start.parents]:
        if (parent / "src").exists():
            return parent
        if (parent / "pyproject.toml").exists():
            return parent
        if (parent / "setup.py").exists():
            return parent

    raise FileNotFoundError(
        "Could not find project root."
    )

# Find project root automatically
PROJECT_ROOT = find_project_root(Path.cwd())

# Add project root to Python path
sys.path.insert(0, str(PROJECT_ROOT))

from config._0_supportfiles.config_loader_optimisation import load_opt_config 
from src.efleetplan._2_co_optimisation.co_optimisation import optimisation, save_results 
from src.efleetplan._2_co_optimisation.optimisation_graphs import graph_vehicles, plot_summary_table, process_folder, graph_number_of_chargers_by_schedules, graph_chargingenergy, graph_energybytype

print(f"PROJECT_ROOT = {PROJECT_ROOT}")

# 2. Charging Infrastructure Co-optimisation Package

## 2.1 Optimisation parameters configurations

### 2.1.1 Main optimisation parameters

In [ ]:
# All parameters are now loaded from YAML configuration files:
#   config/run_Optimisation_Config.yaml   - run settings (schedule, fleet size, solver gap)
#   config/infrastructure_configuration.yaml       - infrastructure power and cost parameters
#
# To change parameters, edit the YAML files directly.
# If infrastructure_configurations is set to "custom" in the run config,
# the custom parameters defined there will override the predefined ones.

config_dir = PROJECT_ROOT / "config"

env, run, predefined = load_config(
    env_yaml       = config_dir / "env.yaml",
    run_yaml       = config_dir / "run_FleetSchedule_Config.yaml",
    predefined_dir = config_dir / "predefined",
)

schedule_name   = run.schedule_name
schedule_number = run.schedule_number

print('Configuration loaded successfully:')
print(f'  Schedule:      {schedule_name} (#{schedule_number})')
print(f'  Fleet size:    {run.EVs} vehicles')
print(f'  MIP Gap:       {run.MIPGap}')

### 2.1.2. Cost and power configuration

In the cost and power configurations above, it is possible to change the values of different paramenters, related to cost of infrastructure, energy subscription rates, route charging rates and battery power and losses.

In [ ]:
# Power and battery parameters (loaded from YAML)
print('--- Chargers Power Parameters ---')
for k, v in power_charge_config.items():
    print(f'  {k}: {v}')

In [ ]:
# Infrastructure cost parameters (loaded from YAML)
print('--- Infrastructure Cost Parameters ---')
for k, v in cost_config.items():
    print(f'  {k}: {v}')

In [ ]:
# Before calling optimisation()
for key in ['En_consumption', 'Ev_distance', 'EV_availability', 'Battery_Limitation', 'PowerRate_Limitation']:
    val = opt_config[key]
    print(f"\n{key}:")
    print(f"  Type:  {type(val).__name__}")
    print(f"  Shape: {val.shape if hasattr(val, 'shape') else 'N/A'}")
    if hasattr(val, 'head'):
        print(f"  Head:\n{val.head(3)}")
    else:
        print(f"  Value: {val}")

In [ ]:
# Check if the index is actually datetime or strings
avail = opt_config['EV_availability']
print(f"Index type: {type(avail.index[0])}")
print(f"Index dtype: {avail.index.dtype}")

## 2.3 Run optimisation

In [ ]:
# Call the optimisation function
m, Price, EV_availability, Distance_km = optimisation(opt_config, cost_config, power_charge_config)

### Saving results

In [ ]:
# User can choose where to save results the results, CSV files

results_folder = PROJECT_ROOT / "data" / "Output" / f"{schedule_name}" / "Results"
os.makedirs(results_folder, exist_ok=True)

csv_file_pathA = results_folder / f"{schedule_number}_Main_variables_results.csv"
csv_file_pathB = results_folder / f"{schedule_number}_results_summary.csv"
csv_file_pathC = results_folder / f"{schedule_number}_results_per_EV.csv"

In [ ]:
save_results(m, Price, EV_availability, Distance_km, csv_file_pathA, csv_file_pathB, csv_file_pathC, cost_config, power_charge_config)

## 2.4. Post-processing and visualisation

In [ ]:
# Set the path to your folder
file_pattern_mean = PROJECT_ROOT / "data" / "Output" / f"{schedule_name}" / "Results" / "*_results_summary.csv"

# Use glob to find the actual file
matched_files = glob(str(file_pattern_mean))
if not matched_files:
    raise FileNotFoundError(f"No file matches the pattern: {file_pattern_mean}")
file_path = matched_files[0]


plot_summary_table(file_path)

### Create maximum and average files

In [ ]:
# Set the path to your folder
folder_path = PROJECT_ROOT / "data" / "Output" / f"{schedule_name}" / "Results"

# Find all files with the pattern *_pertime_averages.csv
filename_pattern = folder_path / "*_Main_variables_results.csv"

process_folder(folder_path, filename_pattern)

In [ ]:
# Set the path to your folder
folder_path = PROJECT_ROOT / "data" / "Output" / f"{schedule_name}" / "Results"
file_pattern_max = PROJECT_ROOT / "data" / "Output" / f"{schedule_name}" / "Results" / "*_max_variable_per_step.csv"
files_pertime_max = glob(str(file_pattern_max))

graph1 = graph_number_of_chargers_by_schedules(folder_path, files_pertime_max)

### Graph 2 - Average charging power and average price of electricity

In [ ]:
# Set the path to your folder
file_pattern_mean = PROJECT_ROOT / "data" / "Output" / f"{schedule_name}" / "Results" / "*_avg_variable_per_step.csv"
# Use glob to find the actual file

matched_files = glob(str(file_pattern_mean))
if not matched_files:
    raise FileNotFoundError(f"No file matches the pattern: {file_pattern_mean}")
file_path = matched_files[0]
folder_path = PROJECT_ROOT / "data" / "Output" / f"{schedule_name}" / "Results"

graph_chargingenergy(file_path, folder_path)

graph_energybytype(file_path, folder_path)

### Graph 3 - Charging, discharging power and SOC

In [ ]:
# Set the path to your folder
file_path = csv_file_pathC

# Set the number of vehicles for the graph
n_vehicles = 3
# Set the number of days for the graph
n_days = 4

graph_vehicles(folder_path, file_path, n_days, n_vehicles)